In [1]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port']

train_df.drop(columns= cols_to_drop, inplace=True, errors='ignore')
valid_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
test_df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

In [2]:
X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

KNN

In [4]:
from sklearn.metrics import f1_score, classification_report
import torch
import numpy as np
import time
import pandas as pd

X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

class TorchKNNClassifier:
    def __init__(self, n_neighbors=5, val_batch_size=256, train_batch_size=50000, device='cuda'):
        self.k = n_neighbors
        self.val_batch_size = val_batch_size
        self.train_batch_size = train_batch_size
        self.device = device
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        # Chuyển dữ liệu lên GPU một lần để tính toán
        if isinstance(X, pd.DataFrame): X = X.values
        if isinstance(y, pd.Series): y = y.values
        self.X_train = torch.tensor(X, dtype=torch.float32, device=self.device)
        self.y_train = torch.tensor(y, dtype=torch.long, device=self.device)

    def predict(self, X):
        if isinstance(X, pd.DataFrame): X = X.values
        X_val = torch.tensor(X, dtype=torch.float32, device=self.device)
        preds = []
        
        # Chia batch dữ liệu CẦN dự đoán 
        for i in range(0, len(X_val), self.val_batch_size):
            X_batch = X_val[i:i+self.val_batch_size]
            
            # Khởi tạo buffer chứa min distances. Mặc định là vô cực (inf)
            current_top_dists = torch.full((len(X_batch), self.k), float('inf'), device=self.device)
            current_top_indices = torch.zeros((len(X_batch), self.k), dtype=torch.long, device=self.device)
            
            # Phân tách X_train thành các khối nhỏ để lặp qua (Tránh tạo ma trận khổng lồ gây OOM)
            for j in range(0, len(self.X_train), self.train_batch_size):
                X_tr_batch = self.X_train[j:j+self.train_batch_size]
                
                # Tính khoảng cách Euclidean
                dists = torch.cdist(X_batch, X_tr_batch)
                
                # Mức độ cắt k
                k_chunk = min(self.k, dists.shape[1])
                chunk_top_dists, chunk_top_idx = torch.topk(dists, k_chunk, dim=1, largest=False)
                
                # Sửa index thành chỉ số toàn cục trong self.X_train
                chunk_top_idx += j
                
                # Gộp với k nhỏ nhất của các khối trước, sau đó lại top-k một lần nữa
                merged_dists = torch.cat([current_top_dists, chunk_top_dists], dim=1)
                merged_idx = torch.cat([current_top_indices, chunk_top_idx], dim=1)
                
                current_top_dists, best_k_pos = torch.topk(merged_dists, self.k, dim=1, largest=False)
                current_top_indices = torch.gather(merged_idx, 1, best_k_pos)

                # Dọn rác VRAM
                del dists, merged_dists, merged_idx
            
            # Lấy nhãn của k láng giềng gần nhất chung cuộc
            topk_labels = self.y_train[current_top_indices]
            
            # Dự đoán theo nhãn chiếm đa số (mode)
            mode_labels, _ = torch.mode(topk_labels, dim=1)
            preds.append(mode_labels.cpu().numpy())
            
        return np.concatenate(preds)

# Thử các giá trị k khác nhau
k_values = [7]

best_k = None
best_f1 = -1
best_model = None

print("Đang huấn luyện phân lớp KNN (Multi-batch Memory Efficient qua PyTorch) và tìm 'k' tốt nhất...")
start_time = time.time()

for k in k_values:
    # Sử dụng TorchKNNClassifier kết hợp cắt nhỏ ma trận, GPU sẽ an toàn
    knn = TorchKNNClassifier(n_neighbors=k, val_batch_size=256, train_batch_size=50000, device='cuda')
    knn.fit(X_train, y_train)
    
    y_valid_pred = knn.predict(X_valid)
    
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"k = {k:2d} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_k = k
        best_model = knn

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> K tốt nhất tìm được là k = {best_k} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test
print("\n--- Đánh giá Model Tốt nhất (k={}) trên tập TEST ---".format(best_k))
y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp KNN (Multi-batch Memory Efficient qua PyTorch) và tìm 'k' tốt nhất...
k =  7 | Validation Macro F1: 0.7890

=> Quá trình huấn luyện và tuning hoàn tất trong 361.07 giây.
=> K tốt nhất tìm được là k = 7 với Validation Macro F1: 0.7890

--- Đánh giá Model Tốt nhất (k=7) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.4233    0.8995    0.5756     19846
           1     0.9793    1.0000    0.9895    484077
           2     0.6667    0.7928    0.7243      2515
           3     0.9913    0.6672    0.7976     36253
           4     0.6604    0.6996    0.6795     18979
           5     0.9490    0.9714    0.9601      2451
           6     0.4564    0.7083    0.5551     11847
           7     1.0000    0.9963    0.9982    107436
           8     0.9174    0.2461    0.3881     16746
           9     0.9999    0.6955    0.8203     41523
          10     0.8195    0.6592    0.7307     18567

    accuracy                         0

SVM

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...")
start_time = time.time()

# Thử các giá trị alpha (L2 regularization)
alpha_values = [0.0001]

best_alpha = None
best_f1 = -1
best_svm_model = None

# Khối 1: Chuyển dữ liệu lên Numpy 
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đẩy sang Tensor CUDA
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
# Đối với Multi-class SVM, giữ nguyên nhãn số nguyên (0, 1, 2... 10)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Kiến trúc Linear SVM (Multi-class) bằng Pytorch
class TorchLinearSVM(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes) # Sửa output thành 11 classes
        
    def forward(self, x):
        return self.linear(x)

# PyTorch hỗ trợ sẵn MultiMarginLoss dùng để tối ưu Multi-class SVM
criterion = nn.MultiMarginLoss()

for alpha in alpha_values:
    model = TorchLinearSVM(input_dim, num_classes).to(device)
    
    # Khai báo Optimizer - sử dụng weight_decay thay thế L2 penalty
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=alpha)
    
    model.train()
    epochs = 100
    batch_size = 8192
    num_samples = X_train_t.shape[0]
    
    for epoch in range(epochs):
        indices = torch.randperm(num_samples, device=device)
        for i in range(0, num_samples, batch_size):
            idx = indices[i:i+batch_size]
            batch_X = X_train_t[idx]
            batch_y = y_train_t[idx]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            # Tính toán trên GPU với Multi-Class Hinge Loss
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
    
    # Validation step
    model.eval()
    with torch.no_grad():
        valid_outputs = model(X_valid_t)
        # Sử dụng argmax để lấy chỉ số có logit cao nhất (tương ứng với nhãn)
        y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
        
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_alpha = alpha
        best_svm_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên test
print("\n--- Đánh giá Model Multi-class SVM Tốt nhất (alpha={}) trên tập TEST ---".format(best_alpha))
best_svm_model.eval()
with torch.no_grad():
    test_outputs = best_svm_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...
alpha = 0.000100 | Validation Macro F1: 0.7435

=> Quá trình huấn luyện và tuning hoàn tất trong 30.85 giây.
=> Cấu hình tốt nhất: alpha = 0.0001 với Validation Macro F1: 0.7435

--- Đánh giá Model Multi-class SVM Tốt nhất (alpha=0.0001) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.4645    0.1287    0.2016     19846
           1     0.9982    1.0000    0.9991    484077
           2     0.6256    0.7901    0.6983      2515
           3     0.9558    0.9308    0.9431     36253
           4     0.5686    0.3065    0.3983     18979
           5     0.9083    0.9543    0.9308      2451
           6     0.6816    0.6331    0.6564     11847
           7     0.9998    0.9964    0.9981    107436
           8     0.4255    0.8190    0.5601     16746
           9     0.9981    0.6814    0.8099     41523
          10     0.4146    0.8902

Random Forest

In [6]:
from xgboost import XGBRFClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử trên các max_depth khác nhau
max_depth_values = [15]

best_depth = None
best_f1 = -1
best_rf_model = None

for depth in max_depth_values:
    # Sử dụng XGBRFClassifier với cấu hình device='cuda' để chạy trên GPU
    rf_model = XGBRFClassifier(
        n_estimators=100, 
        max_depth=depth,
        n_jobs=-1, 
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    rf_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính điểm Macro F1
    y_valid_pred = rf_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_rf_model = rf_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Random Forest Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_rf_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [12:05:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


max_depth = 15 | Validation Macro F1: 0.8889

=> Quá trình huấn luyện và tuning hoàn tất trong 1156.52 giây.
=> Cấu hình tốt nhất: max_depth = 15 với Validation Macro F1: 0.8889

--- Đánh giá mô hình Random Forest Tốt nhất (max_depth=15) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.7819    0.8301    0.8053     19846
           1     0.9983    1.0000    0.9991    484077
           2     0.6842    0.9638    0.8003      2515
           3     0.9975    0.8712    0.9301     36253
           4     0.6828    0.8557    0.7595     18979
           5     0.9203    0.9935    0.9555      2451
           6     0.4163    0.7413    0.5332     11847
           7     1.0000    0.9963    0.9982    107436
           8     0.9453    0.9569    0.9511     16746
           9     1.0000    0.6904    0.8169     41523
          10     0.8158    0.8282    0.8220     18567

    accuracy                         0.9591    760240
   macro avg     0.8402    0.8843    0.8

Decision Tree

In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử nghiệm các giá trị max_depth khác nhau
max_depth_values = [25]

best_depth = None
best_f1 = -1
best_dt_model = None

for depth in max_depth_values:
    # Dùng XGBClassifier với n_estimators=1 để thiết lập giống một Decision Tree chạy bằng GPU
    dt_model = XGBClassifier(
        n_estimators=1,
        learning_rate=1.0,
        max_depth=depth,
        n_jobs=-1,
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    dt_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính Macro F1
    y_valid_pred = dt_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_dt_model = dt_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_dt_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...
max_depth = 25 | Validation Macro F1: 0.8873

=> Quá trình huấn luyện và tuning hoàn tất trong 11.62 giây.
=> Cấu hình tốt nhất: max_depth = 25 với Validation Macro F1: 0.8873

--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth=25) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.7826    0.8331    0.8071     19846
           1     0.9947    1.0000    0.9973    484077
           2     0.7098    0.9606    0.8164      2515
           3     0.9978    0.8715    0.9304     36253
           4     0.6865    0.8456    0.7578     18979
           5     0.9230    0.9833    0.9522      2451
           6     0.4432    0.7458    0.5560     11847
           7     1.0000    0.9963    0.9981    107436
           8     0.9453    0.9623    0.9537     16746
           9     1.0000    0.6885    0.8155     41523
          10     0.8126    0.8220    0.8173     18567



Logistic Regression

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...")
start_time = time.time()

# Thử nghiệm các alpha và penalty l1, l2
alpha_values = [1e-6]
penalty_values = ['l1']

best_alpha = None
best_penalty = None
best_f1 = -1
best_logreg_model = None

# Khối 1: Chuyển đổi toàn bộ mảng dữ liệu về Numpy (nếu đang ở pandas DataFrame)
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đưa thẳng lên VRAM của GPU với PyTorch
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Định nghĩa mạng: 1 tầng Linear duy nhất + Hàm phạt Cross Entropy = Logistic Regression
class TorchLogisticRegression(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes)
        
    def forward(self, x):
        return self.linear(x)

for penalty in penalty_values:
    for alpha in alpha_values:
        model = TorchLogisticRegression(input_dim, num_classes).to(device)
        
        # Nếu l2 thì dùng tham số weight_decay tích hợp sẵn của Optimizer tự tính toán
        weight_decay = alpha if penalty == 'l2' else 0.0
        optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        epochs = 100
        batch_size = 8192
        num_samples = X_train_t.shape[0]
        
        for epoch in range(epochs):
            # Xáo trộn index bộ nhớ trực tiếp trên GPU để đảm bảo tốc độ tối đa
            indices = torch.randperm(num_samples, device=device)
            
            for i in range(0, num_samples, batch_size):
                idx = indices[i:i+batch_size]
                batch_X = X_train_t[idx]
                batch_y = y_train_t[idx]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                # Nếu l1 thì cộng tay penalty L1 Norm
                if penalty == 'l1':
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + alpha * l1_norm
                    
                loss.backward()
                optimizer.step()
        
        model.eval()
        with torch.no_grad():
            valid_outputs = model(X_valid_t)
            y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
            
        current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
        print(f"penalty = {penalty:2s}, alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
        
        # Chọn model tốt nhất dựa trên Macro F1
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_alpha = alpha
            best_penalty = penalty
            best_logreg_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: penalty = {best_penalty}, alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print(f"\n--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty={best_penalty}, alpha={best_alpha}) trên tập TEST ---")
best_logreg_model.eval()
with torch.no_grad():
    test_outputs = best_logreg_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...
penalty = l1, alpha = 0.000001 | Validation Macro F1: 0.7135

=> Quá trình huấn luyện và tuning hoàn tất trong 624.67 giây.
=> Cấu hình tốt nhất: penalty = l1, alpha = 1e-06 với Validation Macro F1: 0.7135

--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty=l1, alpha=1e-06) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.5999    0.1633    0.2567     19846
           1     0.9499    1.0000    0.9743    484077
           2     0.5854    0.7956    0.6745      2515
           3     0.9488    0.8234    0.8816     36253
           4     0.5779    0.3362    0.4251     18979
           5     0.9785    0.9837    0.9811      2451
           6     0.6948    0.6425    0.6676     11847
           7     1.0000    0.6339    0.7759    107436
           8     0.4158    0.8363    0.5554     16746
           9     0.6224    0.6951

In [4]:
import pandas as pd
data_raw = pd.read_csv(r"D:\Downloads\GeneratedLabelledFlows\TrafficLabelling\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")
data_preprocessed = pd.read_csv(r"D:\Downloads\MachineLearningCSV\MachineLearningCVE\Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv")

In [ ]:
print(data_raw.shape)

(286467, 85)


In [6]:
print(data_preprocessed.shape)

(286467, 79)


In [11]:
print(data_raw[" Label"].value_counts())

 Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [12]:
print(data_preprocessed[" Label"].value_counts())

 Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64


In [1]:
import pandas as pd
data = pd.read_csv(r"D:\Downloads\attack_samples_1sec.csv\attack_samples_1sec.csv")


In [2]:
data.shape

(90391, 94)

In [3]:
# chỉnh sao cho in ra toàn bộ nội dung cột của dataframe
pd.set_option('display.max_colwidth', None)

print(data["timestamp"].head(5))

0    2025-01-23T15:31:10.709000Z_2025-01-23T15:31:11.709000Z
1    2025-01-23T15:31:11.709000Z_2025-01-23T15:31:12.709000Z
2    2025-01-23T15:31:12.709000Z_2025-01-23T15:31:13.709000Z
3    2025-01-23T15:31:13.709000Z_2025-01-23T15:31:14.709000Z
4    2025-01-23T15:31:14.709000Z_2025-01-23T15:31:15.709000Z
Name: timestamp, dtype: str


In [8]:
print(data["label2"].value_counts())

label2
recon         33648
dos           18420
ddos          18056
mitm           8062
malware        7541
web            2796
bruteforce     1868
Name: count, dtype: int64


In [1]:
import pandas as pd
data = pd.read_csv(r"D:\Downloads\archive\combined_dataset.csv")

In [2]:
data.shape

(685671, 94)

In [3]:
print(data["label2"].value_counts())

label2
benign        400672
recon         105848
dos            57736
ddos           56692
mitm           25490
malware        24177
web             9040
bruteforce      6016
Name: count, dtype: int64


In [22]:
# ỉn ra index các dòng có label2 = 0
print(data.index[data["label2"] == "bruteforce"].tolist())

[12887, 12888, 12889, 12890, 12891, 12892, 12893, 12894, 12895, 12896, 12897, 12898, 12899, 12900, 12901, 12902, 12903, 12904, 12905, 12906, 12907, 12908, 12909, 12910, 12911, 12912, 12913, 12914, 12915, 12916, 12917, 12918, 12919, 12920, 12921, 12922, 12923, 12924, 12925, 12926, 12927, 12928, 12929, 12930, 12931, 12932, 12933, 12934, 12935, 12936, 12937, 12938, 12939, 12940, 12941, 12942, 12943, 12944, 12945, 13067, 13068, 13069, 13070, 13071, 13072, 13073, 13074, 13075, 13076, 13077, 13078, 13079, 13080, 13081, 13082, 13083, 13084, 13085, 13086, 13087, 13088, 13089, 13090, 13091, 13092, 13093, 13094, 13095, 13096, 13097, 13098, 13099, 13100, 13101, 13102, 13103, 13104, 13105, 13106, 13107, 13108, 13109, 13110, 13111, 13112, 13113, 13114, 13115, 13116, 13117, 13118, 13119, 13120, 13121, 13122, 13123, 13124, 13125, 13159, 13160, 13161, 13162, 13163, 13164, 13165, 13166, 13167, 13168, 13169, 13170, 13171, 13172, 13173, 13174, 13175, 13176, 13177, 13178, 13179, 13180, 13181, 13182, 13183

In [ ]:
print(data.columns)

Index(['device_name', 'device_mac', 'label_full', 'label1', 'label2', 'label3',
       'label4', 'timestamp', 'timestamp_start', 'timestamp_end',
       'log_data-ranges_avg', 'log_data-ranges_max', 'log_data-ranges_min',
       'log_data-ranges_std_deviation', 'log_data-types',
       'log_data-types_count', 'log_interval-messages', 'log_messages_count',
       'network_fragmentation-score', 'network_fragmented-packets',
       'network_header-length_avg', 'network_header-length_max',
       'network_header-length_min', 'network_header-length_std_deviation',
       'network_interval-packets', 'network_ip-flags_avg',
       'network_ip-flags_max', 'network_ip-flags_min',
       'network_ip-flags_std_deviation', 'network_ip-length_avg',
       'network_ip-length_max', 'network_ip-length_min',
       'network_ip-length_std_deviation', 'network_ips_all',
       'network_ips_all_count', 'network_ips_dst', 'network_ips_dst_count',
       'network_ips_src', 'network_ips_src_count', 'network_

In [28]:
# in ra các cột không phải dạng số
import numpy as np
print(data.select_dtypes(exclude=[np.number]).columns.tolist())

['device_name', 'device_mac', 'label_full', 'label1', 'label2', 'label3', 'label4', 'timestamp', 'timestamp_start', 'timestamp_end', 'log_data-types', 'network_ips_all', 'network_ips_dst', 'network_ips_src', 'network_macs_all', 'network_macs_dst', 'network_macs_src', 'network_ports_all', 'network_ports_dst', 'network_ports_src', 'network_protocols_all', 'network_protocols_dst', 'network_protocols_src']


In [30]:
print(data["network_ips_all"].head(5))

0    ['192.168.1.195', '192.168.1.104', '192.168.1.103', '192.168.1.102', '192.168.1.105', '192.168.1.101', '192.168.1.1']
1    ['192.168.1.195', '192.168.1.104', '192.168.1.102', '192.168.1.103', '192.168.1.101', '192.168.1.105', '192.168.1.1']
2                   ['192.168.1.195', '192.168.1.102', '192.168.1.104', '192.168.1.103', '192.168.1.101', '192.168.1.105']
3    ['192.168.1.195', '192.168.1.102', '192.168.1.103', '192.168.1.104', '192.168.1.105', '192.168.1.101', '192.168.1.1']
4    ['192.168.1.195', '192.168.1.103', '192.168.1.102', '192.168.1.105', '192.168.1.104', '192.168.1.101', '192.168.1.1']
Name: network_ips_all, dtype: str


In [18]:
print(data["timestamp"].head(5))

0    2025-01-23T15:31:10.709000Z_2025-01-23T15:31:20.709000Z
1    2025-01-23T15:31:15.709000Z_2025-01-23T15:31:25.709000Z
2    2025-01-23T15:31:20.709000Z_2025-01-23T15:31:30.709000Z
3    2025-01-23T15:31:25.709000Z_2025-01-23T15:31:35.709000Z
4    2025-01-23T15:31:30.709000Z_2025-01-23T15:31:40.709000Z
Name: timestamp, dtype: str
